# Retrieval Extraction Diagnostic

This notebook diagnoses why `extract_structured_with_retrieval` fails while `extract_structured` succeeds.

Steps:
1. Setup
2. Pick a document and inspect what the retriever actually returns
3. Inspect the context block that gets sent to the LLM
4. Run both extraction modes side-by-side and compare outputs
5. Diagnose the root cause

## 1. Setup

In [2]:
import os
import sys
from pathlib import Path

backend_path = Path.cwd().parent.parent / "backend" / "src"
sys.path.insert(0, str(backend_path))

os.environ["DJANGO_SETTINGS_MODULE"] = "uu_backend.django_project.settings.local"
os.environ["DJANGO_DATABASE_URL"] = "postgres://uu:uu@localhost:5432/uu_django"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

print("✓ Django setup complete")

✓ Django setup complete


In [3]:
from uu_backend.django_data.models import DocumentModel
from uu_backend.services.contextual_retrieval.service import get_contextual_retrieval_service
from uu_backend.repositories.document_repository import get_document_repository
from uu_backend.repositories import get_repository
from uu_backend.services.extraction_service import ExtractionService
import pandas as pd

retrieval_service = get_contextual_retrieval_service()
repository = get_repository()
document_repo = get_document_repository()
extraction_service = ExtractionService()

print("✓ Services initialised")

✓ Services initialised


## 2. Pick a Document

In [4]:
# List all indexed documents
docs = DocumentModel.objects.filter(retrieval_index_status='completed').order_by('-created_at')

rows = []
for doc in docs:
    classification = repository.get_classification(str(doc.id))
    doc_type_name = None
    if classification:
        doc_type = repository.get_document_type(classification.document_type_id)
        doc_type_name = doc_type.name if doc_type else None
    rows.append({
        'ID': str(doc.id),
        'Filename': doc.filename,
        'Chunks': doc.retrieval_chunks_count,
        'Doc Type': doc_type_name or '(not classified)',
        'Created': doc.created_at.strftime('%Y-%m-%d %H:%M')
    })

pd.DataFrame(rows)

,ID,Filename,Chunks,Doc Type,Created
0,8269b380-d311-472e-b5ed-d3e4476ba880,ACORD 25 fillable-test-filled-v3.pdf,1,Accord,2026-02-20 03:41


In [5]:
# ── SET THIS ──────────────────────────────────────────────────────────────────
DOCUMENT_ID = "8269b380-d311-472e-b5ed-d3e4476ba880"  # paste from above
# ──────────────────────────────────────────────────────────────────────────────

document = document_repo.get_document(DOCUMENT_ID)
classification = repository.get_classification(DOCUMENT_ID)
doc_type = repository.get_document_type(classification.document_type_id)

print(f"Document : {document.filename}")
print(f"Doc Type : {doc_type.name}")
print(f"Fields   : {[f.name for f in doc_type.schema_fields]}")
print(f"Content length : {len(document.content or '')} chars")

Document : ACORD 25 fillable-test-filled-v3.pdf
Doc Type : Accord
Fields   : ['coverages_table']
Content length : 5630 chars


In [1]:
document.content

NameError: name 'document' is not defined

## 3. Inspect What the Retriever Returns Per Field

In [13]:
build_field_query('test')

AttributeError: 'str' object has no attribute 'name'

In [18]:
field = doc_type.schema_fields[0]

In [19]:
field

SchemaField(name='coverages_table', type=<FieldType.ARRAY: 'array'>, description='Rows from the Coverages table (one object per coverage row)', required=False, extraction_prompt="Extract each row of the COVERAGES table exactly as it appears in the document. Coverages commonly include the type of insurance, policy references, limits, etc. For each row produce the raw text for: coverage type (the 'Type of Insurance' cell), policy number (exact raw string), policy effective date (MM/DD/YYYY if present), policy expiration date (MM/DD/YYYY if present), and the limits column as a list of raw limit lines (each line exactly as shown, do not interpret or convert currency or amounts). Do not normalize, infer, or reformat values — return the raw strings exactly as in the table.", order=None, properties=None, items=SchemaField(name='item', type=<FieldType.OBJECT: 'object'>, description=None, required=False, extraction_prompt=None, order=None, properties={'limits': SchemaField(name='limits', type=<

In [11]:
# Mirror _build_field_query from ExtractionService
def build_field_query(field) -> str:
    parts = [field.name]
    if field.description:
        parts.append(field.description)
    if field.extraction_prompt:
        parts.append(field.extraction_prompt)
    return " ".join(parts)

TOP_K_PER_FIELD = 5

all_chunks = {}
print(f"{'Field':<35} {'Query (first 80 chars)':<80} {'Hits':>4} {'Top Score':>10}")
print("-" * 135)

for field in doc_type.schema_fields:
    query = build_field_query(field)
    results = retrieval_service.search(
        query=query,
        top_k=TOP_K_PER_FIELD,
        filter_doc_id=DOCUMENT_ID,
        use_reranking=True,
    )
    all_chunks[field.name] = results
    top_score = results[0].score if results else 0.0
    print(f"{field.name:<35} {query[:80]:<80} {len(results):>4} {top_score:>10.4f}")

Collection for document 8269b380-d311-472e-b5ed-d3e4476ba880 is empty


Field                               Query (first 80 chars)                                                           Hits  Top Score
---------------------------------------------------------------------------------------------------------------------------------------
coverages_table                     coverages_table Rows from the Coverages table (one object per coverage row) Extr    0     0.0000


In [6]:
# Show the actual text returned for each field
for field_name, results in all_chunks.items():
    print(f"\n{'='*80}")
    print(f"FIELD: {field_name}")
    print(f"{'='*80}")
    if not results:
        print("  ⚠️  NO RESULTS RETURNED")
        continue
    for i, r in enumerate(results, 1):
        print(f"  Result {i} | score={r.score:.4f} | chunk_index={r.chunk_index}")
        print(f"  original_text ({len(r.original_text)} chars):")
        print(f"    {r.original_text[:300].replace(chr(10), ' ')}")
        print()


FIELD: coverages_table
  ⚠️  NO RESULTS RETURNED


## 4. Inspect the Context Block Sent to the LLM

In [ ]:
# Replicate the dedup + context-building logic from extract_structured_with_retrieval
seen_chunk_ids = set()
unique_chunks = []
for field_name, results in all_chunks.items():
    for result in results:
        chunk_id = f"{result.doc_id}_{result.chunk_index}"
        if chunk_id not in seen_chunk_ids:
            seen_chunk_ids.add(chunk_id)
            unique_chunks.append(result)

unique_chunks.sort(key=lambda r: r.score, reverse=True)
top_chunks = unique_chunks[:20]

context_parts = []
for i, chunk in enumerate(top_chunks):
    context_parts.append(f"[Source {i+1}]\n{chunk.original_text}")
context = "\n\n".join(context_parts)

print(f"Total chunks retrieved (all fields) : {sum(len(r) for r in all_chunks.values())}")
print(f"Unique chunks after dedup           : {len(unique_chunks)}")
print(f"Chunks sent as context (max 20)     : {len(top_chunks)}")
print(f"Total context length                : {len(context)} chars")
print(f"Document content length             : {len(document.content or '')} chars")

In [ ]:
# Print the full context block that gets sent to the LLM
print("CONTEXT BLOCK SENT TO LLM:")
print("=" * 80)
print(context)
print("=" * 80)

In [ ]:
# Compare against full document content (what the non-retrieval path uses)
full_content = document.content or ""
if len(full_content) > 8000:
    full_content_used = full_content[:4000] + "\n...[truncated]...\n" + full_content[-4000:]
else:
    full_content_used = full_content

print("FULL DOCUMENT CONTENT (used by non-retrieval path):")
print("=" * 80)
print(full_content_used)
print("=" * 80)

## 5. Side-by-Side Extraction Comparison

In [ ]:
# Run standard extraction (no retrieval) — this should work
print("Running standard extraction (no retrieval)...")
result_standard = extraction_service.extract_structured(DOCUMENT_ID)

print("\n✓ Standard extraction complete")
print(f"Fields extracted: {len(result_standard.fields)}")
for f in result_standard.fields:
    print(f"  {f.field_name}: {str(f.value)[:120]}")

In [ ]:
# Run retrieval extraction — this is the failing path
print("Running retrieval extraction...")
try:
    result_retrieval = extraction_service.extract_structured_with_retrieval(
        DOCUMENT_ID,
        top_k_per_field=TOP_K_PER_FIELD,
    )
    print("\n✓ Retrieval extraction complete")
    print(f"Fields extracted: {len(result_retrieval.fields)}")
    for f in result_retrieval.fields:
        print(f"  {f.field_name}: {str(f.value)[:120]}")
except Exception as e:
    print(f"\n❌ Retrieval extraction FAILED: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Side-by-side diff of field values
standard_map = {f.field_name: f.value for f in result_standard.fields}

try:
    retrieval_map = {f.field_name: f.value for f in result_retrieval.fields}
except NameError:
    retrieval_map = {}
    print("Retrieval extraction failed — retrieval column will be empty")

all_fields = sorted(set(list(standard_map.keys()) + list(retrieval_map.keys())))

rows = []
for field_name in all_fields:
    sv = standard_map.get(field_name)
    rv = retrieval_map.get(field_name)
    match = "✓" if str(sv) == str(rv) else "✗"
    rows.append({
        'Field': field_name,
        'Standard': str(sv)[:100] if sv is not None else 'null',
        'Retrieval': str(rv)[:100] if rv is not None else 'null',
        'Match': match
    })

df = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 120)
print(f"\nMatching fields: {df['Match'].value_counts().get('✓', 0)} / {len(df)}")
df

## 6. Diagnose — What's in the Chunks vs What's Expected

In [ ]:
# Check coverage: for each field that failed, is the answer text present in the retrieved chunks?
def text_in_chunks(value, chunks) -> bool:
    """Check if any part of value appears in retrieved chunks."""
    if value is None:
        return False
    value_str = str(value).lower().strip()
    if not value_str or value_str == 'none':
        return False
    combined = " ".join(r.original_text.lower() for r in chunks)
    # Check for at least a 10-char substring match
    search_str = value_str[:40]
    return search_str in combined

print(f"{'Field':<35} {'Standard Value':<50} {'In Chunks?':>10} {'Chunks Returned':>15}")
print("-" * 115)

for field_name in all_fields:
    sv = standard_map.get(field_name)
    chunks = all_chunks.get(field_name, [])
    found = text_in_chunks(sv, chunks)
    icon = "✓" if found else "✗  ← MISSING"
    sv_str = str(sv)[:48] if sv is not None else 'null'
    print(f"{field_name:<35} {sv_str:<50} {icon:>10} {len(chunks):>15}")

In [ ]:
# Summary diagnostic
print("=" * 60)
print("DIAGNOSTIC SUMMARY")
print("=" * 60)
print(f"Document        : {document.filename}")
print(f"Total chunks    : {doc_type.schema_fields[0].name and document.retrieval_chunks_count}")
print(f"Unique retrieved : {len(unique_chunks)}")
print(f"Context chars   : {len(context)}")
print(f"Full-doc chars  : {len(full_content_used)}")

coverage_hits = sum(
    1 for fn in all_fields
    if text_in_chunks(standard_map.get(fn), all_chunks.get(fn, []))
)
print(f"\nField coverage  : {coverage_hits}/{len(all_fields)} fields found in retrieved chunks")

if len(unique_chunks) == 0:
    print("\n⚠️  ISSUE: No chunks were returned at all. The document may not be indexed.")
elif coverage_hits < len(all_fields):
    print("\n⚠️  ISSUE: Some field values are NOT present in the retrieved chunks.")
    print("   The retriever is selecting the wrong chunks for those fields.")
    print("   Consider: increasing top_k, improving field descriptions/extraction_prompts,")
    print("   or checking chunk size vs document structure.")
else:
    print("\n✓ All expected values appear in retrieved chunks.")
    print("  If extraction still fails, the issue is in the LLM prompt/schema, not retrieval.")